THIS NOTEBOOK IS A DEV FILE: USE THE SCRIPT "EXP-FEW-SHOTS.PY" FOR GENERATION.

Generation - no SFT

In [2]:
import pandas as pd
import pyarrow.parquet as pq

in_filename = "../dataset/qgenllm-ds-seed-123-few-shots.parquet"
df = pq.read_table(in_filename).to_pandas()

seed = in_filename.split("seed-")[1].split("-few")[0]
print("seed is:", seed)
display(df.sample(3))





seed is: 123


,axiom_pattern,axiom,template,question,question_type,op_category,ontology,file,question_type_short,op_category_short,row_id,seed,q_from_os,sample_axioms,sample_questions
124,"[Z] SubclassOf([X]),[Z] SubclassOf([prop] some...","Sell SubclassOf(Distribute),Sell SubclassOf(co...",Which [X/noArticle] [prop/*OP_HAS_NOUNS-third]...,Which distribute has a consequence that is som...,op_What-2-part-1-rel-quant-some,OP_HAS_NOUNS,CopyrightAll,What-2-part-1-rel-quant-some.csv,w2p1r_qs,has_nouns,1456,1699854646,"[olia, AWO, Stuff, Cultural-On, Pizza, Pizza, ...","[ImpersonalVerb SubclassOf(LinguisticConcept),...",[Which linguistic concept has a semantic valen...
303,[X] SubclassOf([prop] some [Y]),PrawnsTopping SubclassOf(hasSpiciness some Mild),True or false: [X] [prop/*OP_HAS_NOUNS-third] ...,True or false: A prawns topping has a spicines...,op_Yes-No-2-part-1-rel+1-quant-some,OP_HAS_NOUNS,Pizza,Yes-No-2-part-1-rel+1-quant-some.csv,yn2p1r_1_qs,has_nouns,86402,1499689557,"[AWO, olia, Cultural-On, Stuff, olia, Cultural...","[Warthog SubclassOf(Participate-In some Loop),...",[A warthog participates in some being a loop. ...
23,"[Z] SubclassOf([X]),[Z] SubclassOf([prop] some...","Warthog SubclassOf(eats some FruitingBody),War...",Which [X/noArticle] [prop/*OP_VERB-third] [Y]?,Which animal eats a fruiting body?,op_What-2-part-1-rel,OP_VERB,AWO,What-2-part-1-rel.csv,w2p1r,verb,189,1052114949,"[Stuff, Pizza, Cultural-On, CopyrightAll, olia...",[Distribution SubclassOf(contiguousPortion onl...,[Which quality contiguouses portion that is a ...


HYPERPARAMETERS HERE

In [3]:
# number of few shot examples to add in the prompt
N_EXAMPLES = 2

# model
model_id = "meta-llama/Llama-3.1-8B-Instruct"

# batch size
BATCH_SIZE = 10

MAX_NEW_TOKENS = 512

Prepare messages as list of dictionaries

In [4]:
# generate the messages
sys_p = "You are a helpful assistant that generates questions in natural language based on OWL axioms provided."

df["few_shots_examples"] = N_EXAMPLES

def build_message(row):
    
    mm = []


    mm.append({"role": "system", "content": sys_p})
    for ax, q in zip(row["sample_axioms"][:N_EXAMPLES], row["sample_questions"][:N_EXAMPLES]):
        user_p = f"Based on the following OWL axiom, generate a question:\nAxiom: {ax}\nQuestion:"
        mm.append({"role": "user", "content": user_p})
        mm.append({"role": "assistant", "content": q})
    # final question
    
    final_user_p = f"Based on the following OWL axiom, generate a question:\nAxiom: {row['axiom']}\nQuestion:"
    mm.append({"role": "user", "content": final_user_p})
    return mm
#<


conversations = []
for i in range(df.shape[0]):
    row = df.iloc[i]
    messages = build_message(row)
    conversations.append(messages)
#<

print(f"Built {len(conversations)} messages with FEW SHOT EXAMPLES set to {N_EXAMPLES}.")
print(conversations[0])

Built 430 messages with FEW SHOT EXAMPLES set to 2.
[{'role': 'system', 'content': 'You are a helpful assistant that generates questions in natural language based on OWL axioms provided.'}, {'role': 'user', 'content': 'Based on the following OWL axiom, generate a question:\nAxiom: Universe SubclassOf(Condition)\nQuestion:'}, {'role': 'assistant', 'content': 'What is universe?'}, {'role': 'user', 'content': 'Based on the following OWL axiom, generate a question:\nAxiom: ThinAndCrispyBase SubclassOf(PizzaBase)\nQuestion:'}, {'role': 'assistant', 'content': 'What is a thin and crispy base?'}, {'role': 'user', 'content': 'Based on the following OWL axiom, generate a question:\nAxiom: Grass SubclassOf(Plant)\nQuestion:'}]


Choose hugging face model here


In [4]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


df["model_id"] = model_id

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Moving model to device: {device}... ", end="")
model.to(device)
print("done")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Moving model to device: cuda... done


In [5]:
def generate_batched(conversations, tokenizer, model, batch_size=10, max_length=4096, max_new_tokens=64):
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    device = next(model.parameters()).device
    print("Using device:", device)
    results = []
    
    print("n conversations:", len(conversations))
    n_batches = (len(conversations) + batch_size - 1) // batch_size
    
    for batch_idx, start in enumerate(range(0, len(conversations), batch_size)):
        print(f"Batch {batch_idx + 1}/{n_batches}")
        batch_messages = conversations[start:start + batch_size]

        enc = tokenizer.apply_chat_template(
            batch_messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
            padding=True,
            truncation=True
        )

        # verify the encoding
        # decoded = tokenizer.decode(
        #     enc["input_ids"][0],
        #     skip_special_tokens=False,  # IMPORTANT: keep special tokens to see the template
        # )
        # print(" -- DECODED --")
        # print(decoded)
        # print(" --")
        
        enc = {k: v.to(device) for k, v in enc.items()}
        # print("encoded conversations moved to device:", device)
        
        # input lengths per sample (exclude left padding)
        padded_prompt_length = enc["input_ids"].shape[1]

        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
        
        for i in range(out.size(0)):
            gen_tokens = out[i, padded_prompt_length:]
            results.append(tokenizer.decode(gen_tokens, skip_special_tokens=True).strip())

    return results


in_convs = conversations  # [:10]
preds = generate_batched(in_convs, tokenizer, model, batch_size=BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS)
df["generated"] = preds

display(df[["question", "generated"]].head())

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Using device: cuda:0
n conversations: 430
Batch 1/43


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Batch 2/43


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Batch 42/43


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Batch 43/43


,question,generated
0,What is grass?,Is grass a type of plant?
1,What is an impala?,Is an impala a terrestrial animal?
2,What is a camelopard?,What is the relationship between a camelopard ...
3,What is a gras?,"What does ""Grass"" refer to in the context of ""..."
4,What is a warthog?,Is a warthog an animal?


Check generated text

In [6]:
import os

out_fld = "gen-results"
os.makedirs(f"../{out_fld}", exist_ok=True)

output_filename = f"qgen2026-results-{model_id.replace('/', '_')}-samples-{N_EXAMPLES}-seed-{seed}.parquet"

df.to_parquet(f"../{out_fld}/{output_filename}", index=False)
print("Saved results to:", f"../{out_fld}/{output_filename}")
print("all done")

Saved results to: ../gen-results/qgen2026-results-meta-llama_Llama-3.1-8B-Instruct-samples-0-seed-123.parquet
all done


Try to free memory

In [7]:
model = None
tokenizer = None
enc = None

import gc
gc.collect()

# empty cuda memory
import torch
torch.cuda.empty_cache()